# YOLO Polyp Detection - 模型效能測試
測試訓練好的 YOLO 模型在 validation set 上的效能

In [2]:
from ultralytics import YOLO
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
import random
import cv2
import seaborn as sns

## 1. 設定路徑與載入模型

In [3]:
# 路徑設定
MODEL_PATH = "/datadrive/polyp/yolo_output/runs/detect/2026_04_01(1)/weights/best.pt"
DATA_YAML  = "/root/Desktop/polyp/yolo_data/od/data.yaml"
VAL_TXT    = "/root/Desktop/polyp/yolo_data/od/val.txt"

# 設備
device = 'mps' if torch.backends.mps.is_available() else ('0' if torch.cuda.is_available() else 'cpu')
print(f"使用設備: {device}")

# 載入模型
model = YOLO(MODEL_PATH)
print(f"模型載入完成: {MODEL_PATH}")

使用設備: 0
模型載入完成: /datadrive/polyp/yolo_output/runs/detect/2026_04_01(1)/weights/best.pt


## 2. Validation Set 整體指標 (mAP, Precision, Recall)

In [4]:
metrics = model.val(
    data=DATA_YAML,
    device=device,
    imgsz=640,
    conf=0.25,
    iou=0.5,
    classes=[0],
    verbose=True,
)

print("\n" + "="*50)
print("驗證集整體指標")
print("="*50)
print(f"mAP@0.5       : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95  : {metrics.box.map:.4f}")
print(f"Precision     : {metrics.box.mp:.4f}")
print(f"Recall        : {metrics.box.mr:.4f}")

Ultralytics 8.4.23 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L40S, 45458MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 217.3±266.5 MB/s, size: 243.0 KB)
val: Scanning /root/Desktop/polyp/yolo_data/od/labels.cache... 3771 images, 1537 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3771/3771 608.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 236/236 7.8it/s 30.4s<0.2s
                   all       3771       2409      0.768      0.746      0.777      0.477
Speed: 0.4ms preprocess, 2.1ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /root/Desktop/polyp/yolo_code/od/runs/detect/val

驗證集整體指標
mAP@0.5       : 0.7766
mAP@0.5:0.95  : 0.4767
Precision     : 0.7678
Recall        : 0.7455


## 3. 訓練曲線（從 results.csv）

In [ ]:
csv_path = Path(MODEL_PATH).parent.parent / "results.csv"
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("訓練曲線", fontsize=14)

plot_cols = [
    ("train/box_loss", "train/cls_loss", "Train Loss"),
    ("val/box_loss",   "val/cls_loss",   "Val Loss"),
    ("metrics/precision(B)", None, "Precision"),
    ("metrics/recall(B)",    None, "Recall"),
    ("metrics/mAP50(B)",     None, "mAP@0.5"),
    ("metrics/mAP50-95(B)",  None, "mAP@0.5:0.95"),
]

for ax, (col1, col2, title) in zip(axes.flat, plot_cols):
    if col1 in df.columns:
        ax.plot(df["epoch"], df[col1], label=col1.split("/")[-1])
    if col2 and col2 in df.columns:
        ax.plot(df["epoch"], df[col2], label=col2.split("/")[-1])
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()
print("已儲存 training_curves.png")

## 4. Confidence Threshold 掃描（PR curve 手動版）

In [ ]:
val_images = Path(VAL_TXT).read_text().strip().splitlines()

thresholds = [0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7]
rows = []

for conf in thresholds:
    m = model.val(
        data=DATA_YAML,
        device=device,
        imgsz=640,
        conf=conf,
        iou=0.5,
        classes=[0],
        verbose=False,
    )
    rows.append({
        "conf": conf,
        "precision": m.box.mp,
        "recall": m.box.mr,
        "mAP50": m.box.map50,
        "mAP50-95": m.box.map,
        "f1": 2 * m.box.mp * m.box.mr / (m.box.mp + m.box.mr + 1e-9),
    })
    print(f"conf={conf:.2f}  P={m.box.mp:.3f}  R={m.box.mr:.3f}  mAP50={m.box.map50:.3f}  F1={rows[-1]['f1']:.3f}")

df_thr = pd.DataFrame(rows)
best_row = df_thr.loc[df_thr["f1"].idxmax()]
print(f"\n最佳 F1 @ conf={best_row['conf']:.2f}  F1={best_row['f1']:.4f}")
df_thr

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df_thr["conf"], df_thr["precision"], 'b-o', label="Precision")
axes[0].plot(df_thr["conf"], df_thr["recall"],    'r-o', label="Recall")
axes[0].plot(df_thr["conf"], df_thr["f1"],        'g-o', label="F1")
axes[0].axvline(best_row["conf"], color='gray', linestyle='--', label=f"Best conf={best_row['conf']}")
axes[0].set_xlabel("Confidence Threshold")
axes[0].set_title("P / R / F1 vs Confidence")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_thr["recall"], df_thr["precision"], 'b-o')
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("PR Curve (Confidence Sweep)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("pr_curve.png", dpi=150)
plt.show()

## 5. 視覺化預測結果（隨機抽樣 val 圖）

In [ ]:
def load_gt_boxes(img_path):
    """讀取對應 label 的 GT bounding boxes (YOLO 格式 → 像素座標)"""
    label_path = Path(str(img_path).replace("/images/", "/labels/")).with_suffix(".txt")
    boxes = []
    if label_path.exists():
        img = Image.open(img_path)
        w, h = img.size
        for line in label_path.read_text().strip().splitlines():
            parts = list(map(float, line.split()))
            cx, cy, bw, bh = parts[1], parts[2], parts[3], parts[4]
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            x2 = (cx + bw / 2) * w
            y2 = (cy + bh / 2) * h
            boxes.append((x1, y1, x2, y2))
    return boxes


CONF_VIS = float(best_row["conf"])  # 使用最佳 conf
N_SHOW   = 12

val_images = Path(VAL_TXT).read_text().strip().splitlines()
sample_imgs = random.sample(val_images, min(N_SHOW, len(val_images)))

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle(f"預測結果 (conf={CONF_VIS:.2f})  藍=GT  紅=Pred", fontsize=13)

for ax, img_path in zip(axes.flat, sample_imgs):
    img = np.array(Image.open(img_path).convert("RGB"))
    results = model.predict(img_path, conf=CONF_VIS, iou=0.5, verbose=False, device=device)
    pred_boxes = results[0].boxes.xyxy.cpu().numpy() if results[0].boxes else []
    pred_confs = results[0].boxes.conf.cpu().numpy() if results[0].boxes else []
    gt_boxes   = load_gt_boxes(img_path)

    ax.imshow(img)
    for (x1, y1, x2, y2) in gt_boxes:
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  linewidth=1.5, edgecolor='blue', facecolor='none')
        ax.add_patch(rect)
    for (x1, y1, x2, y2), conf in zip(pred_boxes, pred_confs):
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  linewidth=1.5, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f"{conf:.2f}", color='red', fontsize=7)
    ax.axis('off')
    ax.set_title(Path(img_path).stem[:20], fontsize=7)

plt.tight_layout()
plt.savefig("predictions_vis.png", dpi=150)
plt.show()

## 6. 每張圖的 TP / FP / FN 統計

In [ ]:
IOU_THR  = 0.5
CONF_EVAL = float(best_row["conf"])

total_tp = total_fp = total_fn = 0
fp_cases = []   # (img_path, gt_boxes, pred_boxes, pred_confs)
fn_cases = []

for img_path in val_images:
    gt    = load_gt_boxes(img_path)
    res   = model.predict(img_path, conf=CONF_EVAL, iou=0.5, verbose=False, device=device)
    preds = res[0].boxes.xyxy.cpu().numpy().tolist() if res[0].boxes else []
    confs = res[0].boxes.conf.cpu().numpy().tolist() if res[0].boxes else []

    matched_gt = set()
    tp = fp = 0
    for p in preds:
        best_iou, best_j = 0, -1
        for j, g in enumerate(gt):
            v = iou_box(p, g)
            if v > best_iou:
                best_iou, best_j = v, j
        if best_iou >= IOU_THR and best_j not in matched_gt:
            tp += 1; matched_gt.add(best_j)
        else:
            fp += 1
    fn = len(gt) - len(matched_gt)
    total_tp += tp; total_fp += fp; total_fn += fn

    if fp > 0:
        fp_cases.append((img_path, gt, preds, confs))
    if fn > 0:
        fn_cases.append((img_path, gt, preds, confs))

precision = total_tp / (total_tp + total_fp + 1e-9)
recall    = total_tp / (total_tp + total_fn + 1e-9)
f1        = 2 * precision * recall / (precision + recall + 1e-9)

print("\n" + "="*50)
print(f"逐圖統計 (conf={CONF_EVAL:.2f}, IoU≥{IOU_THR})")
print("="*50)
print(f"TP={total_tp}  FP={total_fp}  FN={total_fn}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"\nFP 圖片數: {len(fp_cases)}  FN 圖片數: {len(fn_cases)}")

## 7. YOLO 內建訓練圖表

In [ ]:
# YOLO 在訓練時自動儲存的圖表
RUN_DIR = Path(MODEL_PATH).parent.parent

yolo_plots = [
    ("results.png",                        "訓練總覽"),
    ("confusion_matrix.png",               "混淆矩陣"),
    ("confusion_matrix_normalized.png",    "正規化混淆矩陣"),
    ("PR_curve.png",                       "PR Curve"),
    ("F1_curve.png",                       "F1 Curve"),
    ("P_curve.png",                        "Precision Curve"),
    ("R_curve.png",                        "Recall Curve"),
    ("labels.jpg",                         "標籤分佈"),
]

available = [(f, t) for f, t in yolo_plots if (RUN_DIR / f).exists()]
print(f"找到 {len(available)} 張 YOLO 內建圖表")

cols = 3
rows_n = (len(available) + cols - 1) // cols
fig, axes = plt.subplots(rows_n, cols, figsize=(18, 6 * rows_n))
axes_flat = np.array(axes).flat

for ax, (fname, title) in zip(axes_flat, available):
    img = np.array(Image.open(RUN_DIR / fname).convert("RGB"))
    ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

for ax in list(axes_flat)[len(available):]:
    ax.axis('off')

plt.tight_layout()
plt.savefig("yolo_builtin_plots.png", dpi=120)
plt.show()

## 8. 指標長條圖 & 手繪混淆矩陣

In [ ]:
# 指標長條圖
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

labels_ = ["Precision", "Recall", "F1", "mAP@0.5", "mAP@0.5:95"]
values_ = [precision, recall, f1, metrics.box.map50, metrics.box.map]
colors_ = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
bars = axes[0].bar(labels_, values_, color=colors_, edgecolor='white')
axes[0].bar_label(bars, fmt='%.3f', padding=3, fontsize=10)
axes[0].set_ylim(0, 1.15)
axes[0].set_title("模型效能指標")
axes[0].grid(axis='y', alpha=0.3)

# 手繪混淆矩陣 (2x2)
cm = np.array([[total_tp, total_fn],
               [total_fp, 0]])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Pos', 'Pred Neg'],
            yticklabels=['GT Pos', 'GT Neg'],
            ax=axes[1])
axes[1].set_title(f"混淆矩陣 (conf={CONF_EVAL:.2f}, IoU≥{IOU_THR})")
axes[1].set_ylabel("Ground Truth")
axes[1].set_xlabel("Prediction")

plt.tight_layout()
plt.savefig("metrics_and_cm.png", dpi=150)
plt.show()

## 9. False Positive 失誤案例

In [ ]:
N_FP = min(8, len(fp_cases))
if N_FP == 0:
    print("沒有 FP 案例！")
else:
    sample_fp = random.sample(fp_cases, N_FP)
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    fig.suptitle("False Positive 案例（模型多偵測）  藍=GT  綠=TP Pred  紅=FP Pred", fontsize=11, color='darkorange')

    for ax, (img_path, gt_boxes, preds, confs) in zip(axes.flat, sample_fp):
        img = np.array(Image.open(img_path).convert("RGB"))
        ax.imshow(img)
        for (x1, y1, x2, y2) in gt_boxes:
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor='#3399FF', facecolor='none'))
        for (x1, y1, x2, y2), c in zip(preds, confs):
            matched = any(iou_box((x1,y1,x2,y2), g) >= IOU_THR for g in gt_boxes)
            color = '#00CC44' if matched else '#FF4444'
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor=color, facecolor='none'))
            ax.text(x1, max(y1-4, 0), f"{c:.2f}", color=color, fontsize=7, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.5, pad=1, edgecolor='none'))
        ax.set_title(f"{Path(img_path).stem[:18]}\nGT={len(gt_boxes)} Pred={len(preds)}", fontsize=7)
        ax.axis('off')

    for ax in list(axes.flat)[N_FP:]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig("fp_cases.png", dpi=150)
    plt.show()

## 10. False Negative 失誤案例

In [ ]:
N_FN = min(8, len(fn_cases))
if N_FN == 0:
    print("沒有 FN 案例！")
else:
    sample_fn = random.sample(fn_cases, N_FN)
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    fig.suptitle("False Negative 案例（模型漏偵測）  藍=偵測到 GT  橘=漏偵測 GT  紅=Pred", fontsize=11, color='red')

    for ax, (img_path, gt_boxes, preds, confs) in zip(axes.flat, sample_fn):
        img = np.array(Image.open(img_path).convert("RGB"))
        ax.imshow(img)

        matched_gt = set()
        for p in preds:
            for j, g in enumerate(gt_boxes):
                if iou_box(p, g) >= IOU_THR and j not in matched_gt:
                    matched_gt.add(j)

        for j, (x1, y1, x2, y2) in enumerate(gt_boxes):
            color = '#3399FF' if j in matched_gt else '#FF8800'
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                         linewidth=2.5, edgecolor=color, facecolor='none'))
            if j not in matched_gt:
                ax.text((x1+x2)/2, (y1+y2)/2, "MISS", ha='center', va='center',
                        color='#FF8800', fontsize=8, fontweight='bold',
                        bbox=dict(facecolor='white', alpha=0.6, pad=1, edgecolor='none'))

        for (x1, y1, x2, y2), c in zip(preds, confs):
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor='#FF4444', facecolor='none'))
            ax.text(x1, max(y1-4, 0), f"{c:.2f}", color='#FF4444', fontsize=7, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.5, pad=1, edgecolor='none'))

        ax.set_title(f"{Path(img_path).stem[:18]}\nGT={len(gt_boxes)} Pred={len(preds)}", fontsize=7)
        ax.axis('off')

    for ax in list(axes.flat)[N_FN:]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig("fn_cases.png", dpi=150)
    plt.show()

## 11. GT Box 大小分佈（TP vs FN）

In [ ]:
# 收集各 GT box 面積，分 TP / FN
hit_areas  = []
miss_areas = []

for img_path in val_images:
    gt = load_gt_boxes(img_path)
    if not gt:
        continue
    res   = model.predict(img_path, conf=CONF_EVAL, iou=0.5, verbose=False, device=device)
    preds = res[0].boxes.xyxy.cpu().numpy().tolist() if res[0].boxes else []

    for j, g in enumerate(gt):
        area = (g[2]-g[0]) * (g[3]-g[1])
        matched = any(iou_box(p, g) >= IOU_THR for p in preds)
        (hit_areas if matched else miss_areas).append(area)

all_areas = hit_areas + miss_areas
bins = np.linspace(0, np.percentile(all_areas, 98) if all_areas else 1, 40)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(hit_areas,  bins=bins, alpha=0.6, color='green', label=f"TP ({len(hit_areas)})")
axes[0].hist(miss_areas, bins=bins, alpha=0.6, color='red',   label=f"FN ({len(miss_areas)})")
axes[0].set_xlabel("Box Area (pixels²)")
axes[0].set_ylabel("Count")
axes[0].set_title("GT Box 面積分佈（TP vs FN）")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for arr, label, color in [(hit_areas, "TP", 'green'), (miss_areas, "FN", 'red')]:
    if arr:
        sa  = np.sort(arr)
        cdf = np.arange(1, len(sa)+1) / len(sa)
        axes[1].plot(sa, cdf, label=label, color=color)
axes[1].set_xlabel("Box Area (pixels²)")
axes[1].set_ylabel("CDF")
axes[1].set_title("Box 面積 CDF（TP vs FN）")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("box_size_dist.png", dpi=150)
plt.show()

## 12. 摘要表格

In [ ]:
summary = pd.DataFrame([{
    "Model": Path(MODEL_PATH).parts[-3],
    "Best conf": CONF_EVAL,
    "mAP@0.5": round(metrics.box.map50, 4),
    "mAP@0.5:0.95": round(metrics.box.map, 4),
    "Precision": round(precision, 4),
    "Recall": round(recall, 4),
    "F1": round(f1, 4),
    "TP": total_tp,
    "FP": total_fp,
    "FN": total_fn,
}])

display(summary)
summary.to_csv("eval_summary.csv", index=False)
print("已儲存 eval_summary.csv")